[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-2/chatbot-summarization.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58239436-lesson-5-chatbot-w-summarizing-messages-and-memory)

# 带消息摘要的聊天机器人

## 回顾

我们已经介绍了如何自定义图状态 Schema 和 reducer。

我们还展示了几种在状态中裁剪或过滤消息的方法。

## 目标

现在，让我们更进一步！

不仅仅是裁剪或过滤消息，我们将展示如何使用 LLM 生成对话的运行摘要。

这使我们能够保留完整对话的压缩表示，而不是通过裁剪或过滤简单地删除它。

我们将把这种摘要功能整合到一个简单的聊天机器人中。

并且我们将为聊天机器人配备记忆功能，支持长时间的对话而不会产生高额的 token 成本/延迟。

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_deepseek

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("DEEPSEEK_API_KEY")

我们将使用 [LangSmith](https://docs.langchain.com/langsmith/home) 进行[追踪](https://docs.langchain.com/langsmith/observability-concepts)。

In [ ]:
_set_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langchain-academy"

In [ ]:
from langchain_deepseek import ChatDeepSeek
model = ChatDeepSeek(model="deepseek-chat",temperature=0)

和之前一样，我们将使用 `MessagesState`。

除了内置的 `messages` 键之外，我们现在还将包含一个自定义键（`summary`）。

In [ ]:
from langgraph.graph import MessagesState
class State(MessagesState):
    summary: str

我们将定义一个节点来调用我们的 LLM，如果存在摘要，则将其整合到 prompt 中。

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, RemoveMessage

# 定义调用模型的逻辑
def call_model(state: State):
    
    # 获取摘要（如果存在）
    summary = state.get("summary", "")

    # 如果有摘要，则将其添加进去
    if summary:
        
        # 将摘要添加到系统消息中
        system_message = f"Summary of conversation earlier: {summary}"

        # 将摘要追加到较新的消息之前
        messages = [SystemMessage(content=system_message)] + state["messages"]
    
    else:
        messages = state["messages"]
    
    response = model.invoke(messages)
    return {"messages": response}

我们将定义一个节点来生成摘要。

注意，这里我们将使用 `RemoveMessage` 在生成摘要后过滤状态。

In [ ]:
def summarize_conversation(state: State):
    
    # 首先，获取任何现有的摘要
    summary = state.get("summary", "")

    # 创建我们的摘要 prompt
    if summary:
        
        # 已经存在一个摘要
        summary_message = (
            f"This is summary of the conversation to date: {summary}\n\n"
            "Extend the summary by taking into account the new messages above:"
        )
        
    else:
        summary_message = "Create a summary of the conversation above:"

    # 将 prompt 添加到对话历史中
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 删除除最近 2 条消息之外的所有消息
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

我们将添加一个条件边，根据对话长度来决定是否生成摘要。

In [ ]:
from langgraph.graph import END
from typing_extensions import Literal
# 判断是结束还是进行摘要
def should_continue(state: State) -> Literal ["summarize_conversation",END]:
    
    """返回要执行的下一个节点。"""
    
    messages = state["messages"]
    
    # 如果消息超过六条，则对对话进行摘要
    if len(messages) > 6:
        return "summarize_conversation"
    
    # 否则可以直接结束
    return END

## 添加记忆

回想一下，[状态是瞬态的](https://github.com/langchain-ai/langgraph/discussions/352#discussioncomment-9291220)，仅限于单次图执行。

这限制了我们进行带有中断的多轮对话的能力。

正如模块 1 末尾介绍的，我们可以使用[持久化](https://docs.langchain.com/oss/python/langgraph/persistence)来解决这个问题！

LangGraph 可以使用检查点器在每个步骤后自动保存图状态。

这个内置的持久化层提供了记忆功能，允许 LangGraph 从上次状态更新处恢复。

正如我们之前展示的，最容易使用的是 `MemorySaver`，一个用于图状态的内存键值存储。

我们需要做的只是用检查点器编译图，我们的图就拥有了记忆！

In [ ]:
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START

# 定义一个新图
workflow = StateGraph(State)
workflow.add_node("conversation", call_model)
workflow.add_node(summarize_conversation)

# 设置入口点为 conversation
workflow.add_edge(START, "conversation")
workflow.add_conditional_edges("conversation", should_continue)
workflow.add_edge("summarize_conversation", END)

# 编译
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

## Threads（线程）

检查点器将每个步骤的状态保存为一个检查点。

这些保存的检查点可以分组到一个对话`thread`中。

可以类比 Slack：不同的频道承载不同的对话。

Threads 就像 Slack 频道，捕获分组的状态集合（例如，对话）。

在下面，我们使用 `configurable` 来设置线程 ID。

![state.jpg](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbadf3b379c2ee621adfd1_chatbot-summarization1.png)

In [ ]:
# 创建一个线程
config = {"configurable": {"thread_id": "1"}}

# 开始对话
input_message = HumanMessage(content="hi! I'm Lance")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="what's my name?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

input_message = HumanMessage(content="i like the 49ers!")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

现在，我们还没有状态的摘要，因为我们仍然有 <= 6 条消息。

这是在 `should_continue` 中设置的。

```
    # 如果消息超过六条，则对对话进行摘要
    if len(messages) > 6:
        return "summarize_conversation"
```

我们可以继续对话，因为我们拥有这个线程。

In [ ]:
graph.get_state(config).values.get("summary","")

带有线程 ID 的 `config` 允许我们从之前记录的状态继续！

In [ ]:
input_message = HumanMessage(content="i like Nick Bosa, isn't he the highest paid defensive player?")
output = graph.invoke({"messages": [input_message]}, config) 
for m in output['messages'][-1:]:
    m.pretty_print()

In [ ]:
graph.get_state(config).values.get("summary","")

## LangSmith

让我们查看追踪记录！